In [2]:
# xlsxwriter = Excel formatting & charts ku use aagura library
!pip install xlsxwriter -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 3.6 MB/s eta 0:00:00


In [3]:
# Colab la file upload panna helper
from google.colab import files

# Upload button varum, healthcare_dataset.csv select pannu
uploaded = files.upload()

Saving healthcare_dataset.csv to healthcare_dataset.csv


In [4]:
# pandas = data handling, numpy = number operations
import pandas as pd
import numpy as np

# upload pannina file oda name eduthukrom
file_name = list(uploaded.keys())[0]

# CSV read pannrom
df = pd.read_csv(file_name, encoding='latin-1')

# rows & columns count
print("Rows, Columns:", df.shape)

# first 5 rows display
df.head()

Rows, Columns: (55500, 15)


,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal


In [5]:
# column names, data types, null count
print(df.info())

# ovvoru column la missing values
print("\nMissing values:\n", df.isnull().sum())

# duplicate rows count
print("\nDuplicate rows:", df.duplicated().sum())

# numeric columns min, max, average (outlier paaka)
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55500 entries, 0 to 55499
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Name                55500 non-null  object 
 1   Age                 55500 non-null  int64  
 2   Gender              55500 non-null  object 
 3   Blood Type          55500 non-null  object 
 4   Medical Condition   55500 non-null  object 
 5   Date of Admission   55500 non-null  object 
 6   Doctor              55500 non-null  object 
 7   Hospital            55500 non-null  object 
 8   Insurance Provider  55500 non-null  object 
 9   Billing Amount      55500 non-null  float64
 10  Room Number         55500 non-null  int64  
 11  Admission Type      55500 non-null  object 
 12  Discharge Date      55500 non-null  object 
 13  Medication          55500 non-null  object 
 14  Test Results        55500 non-null  object 
dtypes: float64(1), int64(2), object(12)
memory usage: 6.4

,Age,Billing Amount,Room Number
count,55500.000000,55500.000000,55500.000000
mean,51.539459,25539.316097,301.134829
std,19.602454,14211.454431,115.243069
min,13.000000,-2008.492140,101.000000
25%,35.000000,13241.224652,202.000000
50%,52.000000,25538.069376,302.000000
75%,68.000000,37820.508436,401.000000
max,89.000000,52764.276736,500.000000


In [6]:
# cleaning munnadi original row count save pannrom (data quality log ku)
raw_rows = len(df)

# column names la space & hyphen remove (Date of Admission -> Date_of_Admission)
df.columns = df.columns.str.strip().str.replace(' ', '_').str.replace('-', '_')

# fully duplicate rows remove pannrom
df = df.drop_duplicates()
duplicates_removed = raw_rows - len(df)

# text columns la extra spaces trim pannrom
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# upper/lower case mess fix: Title Case ku maathrom
for col in ['Doctor', 'Hospital', 'Medical_Condition', 'Admission_Type', 'Insurance_Provider']:
    df[col] = df[col].str.title()

# ---------- HOSPITAL NAME CLEANING ----------
# "Hoover, And" maari suffix remove pannrom
df['Hospital'] = df['Hospital'].str.replace(r'(,\s*|\s+)And$', '', regex=True)

# "And Mahoney" maari prefix remove pannrom
df['Hospital'] = df['Hospital'].str.replace(r'^And\s+', '', regex=True)

# ella commas um remove pannrom
df['Hospital'] = df['Hospital'].str.replace(',', '', regex=False)

# "Ltd Schroeder" -> "Schroeder Ltd" (company suffix last ku move pannrom)
df['Hospital'] = df['Hospital'].str.replace(r'^(Ltd|Inc|Plc|Llc|Group)\s+(.*)$', r'\2 \1', regex=True)

# extra spaces trim pannrom
df['Hospital'] = df['Hospital'].str.strip()

# ---------- DOCTOR NAME CLEANING ----------
# Mr./Mrs./Dr. prefix remove pannrom
df['Doctor'] = df['Doctor'].str.replace(r'^(Mr|Mrs|Ms|Miss|Dr)\.?\s+', '', regex=True)

# Jr/Md/Dds/Phd/Dvm maari suffix remove pannrom
df['Doctor'] = df['Doctor'].str.replace(r'(\s+(Jr|Sr|Md|Dds|Phd|Dvm|Ii|Iii|Iv)\.?)+$', '', regex=True)

# PRIVACY: patient name report la vendaam, remove pannrom
df = df.drop(columns=['Name'])

# date columns ah proper date format ku maathrom
df['Date_of_Admission'] = pd.to_datetime(df['Date_of_Admission'], errors='coerce')
df['Discharge_Date'] = pd.to_datetime(df['Discharge_Date'], errors='coerce')

# invalid date rows count pannitu remove pannrom
invalid_dates = int(df[['Date_of_Admission', 'Discharge_Date']].isnull().any(axis=1).sum())
df = df.dropna(subset=['Date_of_Admission', 'Discharge_Date'])

# Billing & Age numeric ah maathrom
df['Billing_Amount'] = pd.to_numeric(df['Billing_Amount'], errors='coerce')
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')

# missing numeric rows count pannitu remove pannrom
missing_numeric = int(df[['Billing_Amount', 'Age']].isnull().any(axis=1).sum())
df = df.dropna(subset=['Billing_Amount', 'Age'])

# negative / zero billing invalid, remove pannrom
invalid_billing = int((df['Billing_Amount'] <= 0).sum())
df = df[df['Billing_Amount'] > 0]

# discharge date, admission date ku munnadi irundha logic error, remove pannrom
invalid_los = int((df['Discharge_Date'] < df['Date_of_Admission']).sum())
df = df[df['Discharge_Date'] >= df['Date_of_Admission']]

# helper columns: Year_Month, Length of Stay, Age Group
df['Year_Month'] = df['Date_of_Admission'].dt.to_period('M').astype(str)            # example: 2024-04
df['Length_of_Stay'] = (df['Discharge_Date'] - df['Date_of_Admission']).dt.days     # ethana naal admit
df['Age_Group'] = pd.cut(df['Age'], bins=[0, 18, 35, 50, 65, 120],
                         labels=['0-18', '19-35', '36-50', '51-65', '65+'])         # age buckets
df['Age_Group'] = df['Age_Group'].astype(str)                                       # string ah maathrom

# cleaning summary print
print("Raw rows:", raw_rows)
print("Duplicates removed:", duplicates_removed)
print("Invalid dates removed:", invalid_dates)
print("Missing numeric rows removed:", missing_numeric)
print("Invalid billing removed:", invalid_billing)
print("Discharge before admission removed:", invalid_los)
print("Final clean rows:", len(df))

# hospital names la innum comma / "And" irukka nu check (0 vandha clean)
print("Hospital names still messy:", int(df['Hospital'].str.contains(r',|\sAnd$').sum()))

# ovvoru month la admissions count (partial months spot panna)
print(df['Year_Month'].value_counts().sort_index().head(3))
print(df['Year_Month'].value_counts().sort_index().tail(3))

Raw rows: 55500
Duplicates removed: 534
Invalid dates removed: 0
Missing numeric rows removed: 0
Invalid billing removed: 106
Discharge before admission removed: 0
Final clean rows: 54860
Hospital names still messy: 0
Year_Month
2019-05    672
2019-06    897
2019-07    950
Name: count, dtype: int64
Year_Month
2024-03    900
2024-04    939
2024-05    212
Name: count, dtype: int64


In [11]:
# ---------- PARTIAL MONTHS AUTOMATIC DETECT ----------
# ovvoru month la admissions count
month_counts = df.groupby('Year_Month').size()

# median admissions find pannrom
median_adm = month_counts.median()

# median la 80% kku kammi ah irukkura months = partial months (data mudiyaama nindra months)
partial_months = month_counts[month_counts < 0.8 * median_adm].index.tolist()

# partial illaadha complete months list
complete_months = [m for m in sorted(month_counts.index) if m not in partial_months]

# ---------- REPORT MONTH SETTING ----------
# default ah last COMPLETE month (2024-04)
report_month = complete_months[-1]
# vera month vennum na meela line ku keela idha uncomment panni maathu:
#report_month = '2024-03'

# previous complete month (MoM compare ku); first month na None
idx = complete_months.index(report_month)
prev_month = complete_months[idx - 1] if idx > 0 else None

# current month data & previous month data separate pannrom
cur = df[df['Year_Month'] == report_month]
prev = df[df['Year_Month'] == prev_month] if prev_month else df.iloc[0:0]

# ---------- KPI FUNCTION ----------
# oru month data kudutha, 6 KPIs return panra function
def kpis(d):
    admissions = len(d)                                                                    # total admissions (1 row = 1 admission)
    billing = d['Billing_Amount'].sum()                                                    # total billing
    avg_bill = (billing / admissions) if admissions else 0                                 # avg billing per admission
    avg_los = d['Length_of_Stay'].mean() if admissions else 0                              # average length of stay (days)
    emergency = ((d['Admission_Type'] == 'Emergency').mean() * 100) if admissions else 0   # emergency admissions %
    abnormal = ((d['Test_Results'] == 'Abnormal').mean() * 100) if admissions else 0       # abnormal test results %
    return admissions, billing, avg_bill, avg_los, emergency, abnormal

# current & previous month KPIs calculate pannrom
c = kpis(cur)
p = kpis(prev)

# MoM growth % function (previous zero na 0 return pannum)
def growth(now, before):
    return round((now - before) / before * 100, 2) if before else 0

# ---------- SUMMARY KPI TABLE ----------
kpi_df = pd.DataFrame({
    'KPI': ['Total Admissions', 'Total Billing', 'Avg Billing per Admission',
            'Avg Length of Stay (days)', 'Emergency Admissions %', 'Abnormal Test Results %'],
    'Current Month': [round(x, 2) for x in c],
    'Previous Month': [round(x, 2) for x in p],
    'MoM Change %': [growth(c[i], p[i]) for i in range(6)]
})

# percentage KPIs (Emergency %, Abnormal %) ku relative % badhila percentage point difference
for i in [4, 5]:
    kpi_df.loc[i, 'MoM Change %'] = round(c[i] - p[i], 2)

# ovvoru row oda change unit (% or pp)
kpi_df['Change Unit'] = ['%', '%', '%', '%', 'pp', 'pp']

# ---------- DETAIL SUMMARIES (current month) ----------
# Medical condition wise: admissions, billing, avg length of stay
condition_df = (cur.groupby('Medical_Condition')
                .agg(Admissions=('Age', 'count'), Billing=('Billing_Amount', 'sum'),
                     Avg_LOS=('Length_of_Stay', 'mean'))
                .round(2).reset_index().sort_values('Billing', ascending=False))

# Admission type wise (Emergency / Urgent / Elective)
admission_df = (cur.groupby('Admission_Type')
                .agg(Admissions=('Age', 'count'), Billing=('Billing_Amount', 'sum'),
                     Avg_LOS=('Length_of_Stay', 'mean'))
                .round(2).reset_index().sort_values('Admissions', ascending=False))

# Insurance provider wise (billing source analysis)
insurance_df = (cur.groupby('Insurance_Provider')
                .agg(Admissions=('Age', 'count'), Billing=('Billing_Amount', 'sum'))
                .round(2).reset_index().sort_values('Billing', ascending=False))

# Age group wise
age_df = (cur.groupby('Age_Group')
          .agg(Admissions=('Age', 'count'), Billing=('Billing_Amount', 'sum'))
          .round(2).reset_index())

# Medication wise
medication_df = (cur.groupby('Medication')
                 .agg(Admissions=('Age', 'count'), Billing=('Billing_Amount', 'sum'))
                 .round(2).reset_index().sort_values('Admissions', ascending=False))

# Medical condition wise abnormal test results %
abnormal_df = (cur.assign(Is_Abnormal=(cur['Test_Results'] == 'Abnormal'))
               .groupby('Medical_Condition')['Is_Abnormal'].mean().mul(100).round(2)
               .reset_index(name='Abnormal_Test_Pct').sort_values('Abnormal_Test_Pct', ascending=False))

# ---------- MONTHLY TREND (complete months mattum) ----------
monthly_df = (df.groupby('Year_Month')
              .agg(Admissions=('Age', 'count'), Billing=('Billing_Amount', 'sum'))
              .round(2).reset_index())

# partial months ah trend chart la irundhu remove pannrom
monthly_df = monthly_df[~monthly_df['Year_Month'].isin(partial_months)].reset_index(drop=True)

# ---------- DATA QUALITY LOG ----------
dq_df = pd.DataFrame({
    'Check': ['Raw rows', 'Duplicates removed', 'Invalid dates removed',
              'Missing numeric rows removed', 'Invalid billing removed',
              'Discharge before admission removed', 'Final clean rows'],
    'Count': [raw_rows, duplicates_removed, invalid_dates, missing_numeric,
              invalid_billing, invalid_los, len(df)]
})

# partial months rows count pannitu data quality log la add pannrom
partial_rows = int(df['Year_Month'].isin(partial_months).sum())
dq_df.loc[len(dq_df)] = ['Partial months excluded from trend: ' + ', '.join(partial_months), partial_rows]

# ---------- QUICK CHECK ----------
print("Report month:", report_month, "| Compared with:", prev_month)
print("Partial months excluded:", partial_months)

# KPI table display pannrom
kpi_df

Report month: 2024-04 | Compared with: 2024-03
Partial months excluded: ['2019-05', '2024-05']


,KPI,Current Month,Previous Month,MoM Change %,Change Unit
0,Total Admissions,939.00,900.00,4.33,%
1,Total Billing,23408072.64,22779057.62,2.76,%
2,Avg Billing per Admission,24928.72,25310.06,-1.51,%
3,Avg Length of Stay (days),16.19,15.99,1.28,%
4,Emergency Admissions %,35.14,32.00,3.14,pp
5,Abnormal Test Results %,31.84,32.11,-0.27,pp


In [12]:
# insight lines store panna empty list
insights = []

# 1. Admissions trend: up/down, evlo %, evlo admissions
adm_dir = 'up' if c[0] >= p[0] else 'down'
insights.append(f"Admissions {adm_dir} {abs(growth(c[0], p[0])):.1f}% MoM to {int(c[0])} (vs {int(p[0])} in {prev_month}).")

# 2. Billing vs Avg billing: growth volume-driven ah rate-driven ah nu check pannrom
bill_dir = 'rose' if c[1] >= p[1] else 'fell'
avg_dir = 'rose' if c[2] >= p[2] else 'fell'
bill_text = f"Total billing {bill_dir} {abs(growth(c[1], p[1])):.1f}% MoM; average billing per admission {avg_dir} {abs(growth(c[2], p[2])):.1f}%."
# billing up aana avg billing down na, growth admissions count nala nu note pannrom
if c[1] > p[1] and c[2] < p[2]:
    bill_text += " Billing growth was volume-driven, not rate-driven."
insights.append(bill_text)

# 3. Emergency share: percentage point change
em_dir = 'rose' if c[4] >= p[4] else 'fell'
insights.append(f"Emergency admission share {em_dir} {abs(c[4] - p[4]):.1f} pp to {c[4]:.1f}%.")

# 4. Average length of stay
los_dir = 'increased' if c[3] >= p[3] else 'decreased'
insights.append(f"Average length of stay {los_dir} to {c[3]:.1f} days (from {p[3]:.1f}).")

# 5. Top medical condition by billing (condition_df billing la sort aagirukku)
top_cond = condition_df.iloc[0]
share = top_cond['Billing'] / condition_df['Billing'].sum() * 100
insights.append(f"Top condition by billing: {top_cond['Medical_Condition']} ({share:.1f}% of total billing).")

# 6. Top insurance provider by billing
top_ins = insurance_df.iloc[0]
ins_share = top_ins['Billing'] / insurance_df['Billing'].sum() * 100
insights.append(f"Largest billing source: {top_ins['Insurance_Provider']} ({ins_share:.1f}% of total billing).")

# 7. Abnormal test results share (pp change)
ab_dir = 'rose' if c[5] >= p[5] else 'fell'
insights.append(f"Abnormal test result share {ab_dir} {abs(c[5] - p[5]):.1f} pp to {c[5]:.1f}%.")

# print pannitu paaru
for i in insights:
    print("-", i)

- Admissions up 4.3% MoM to 939 (vs 900 in 2024-03).
- Total billing rose 2.8% MoM; average billing per admission fell 1.5%. Billing growth was volume-driven, not rate-driven.
- Emergency admission share rose 3.1 pp to 35.1%.
- Average length of stay increased to 16.2 days (from 16.0).
- Top condition by billing: Arthritis (18.5% of total billing).
- Largest billing source: Cigna (24.1% of total billing).
- Abnormal test result share fell 0.3 pp to 31.8%.


In [13]:
# Excel column letter find panra helper (Validation formulas ku)
from xlsxwriter.utility import xl_col_to_name

# output file name; month automatic ah varum
output_file = f"Hospital_MIS_Report_{report_month}.xlsx"

# ExcelWriter open pannrom (datetime_format = dates la time varaadhu)
with pd.ExcelWriter(output_file, engine='xlsxwriter', datetime_format='yyyy-mm-dd') as writer:
    wb = writer.book  # workbook object

    # ---------- FORMATS ----------
    # header style: dark blue background, white bold text
    header_fmt = wb.add_format({'bold': True, 'bg_color': '#1F4E78',
                                'font_color': 'white', 'border': 1, 'align': 'center'})
    # title style
    title_fmt = wb.add_format({'bold': True, 'font_size': 16})
    # section heading style
    section_fmt = wb.add_format({'bold': True, 'font_size': 12, 'font_color': '#1F4E78'})
    # red font (negative MoM highlight ku)
    red_fmt = wb.add_format({'font_color': '#C00000', 'bold': True})
    # number formats: integer, and comma + 2 decimals
    int_fmt = wb.add_format({'num_format': '#,##0'})
    dec_fmt = wb.add_format({'num_format': '#,##0.00'})

    # ---------- HELPER FUNCTION ----------
    # DataFrame ah sheet la ezhudhi, header style + width + Billing column format set pannum
    def write_table(data, sheet_name, startrow=0):
        data.to_excel(writer, sheet_name=sheet_name, index=False, startrow=startrow)   # data write
        ws = writer.sheets[sheet_name]                                                # sheet object
        for i, col in enumerate(data.columns):
            ws.write(startrow, i, col, header_fmt)                                    # header style
            width = max(len(str(col)), data[col].astype(str).str.len().max()) + 2     # auto width
            fmt = dec_fmt if 'Billing' in col else None                               # Billing column ku comma format
            ws.set_column(i, i, min(width, 45), fmt)                                  # width + format apply
        return ws

    # ---------- Sheet 1: Summary ----------
    kpi_out = kpi_df.rename(columns={'MoM Change %': 'MoM Change'})        # header la % remove (unit column irukku)
    ws = write_table(kpi_out, 'Summary', startrow=3)
    ws.write(0, 0, f"Hospital MIS Report - {report_month}", title_fmt)    # title
    ws.write(1, 0, f"Compared with: {prev_month}")                          # comparison month
    ws.conditional_format('D5:D10', {'type': 'cell', 'criteria': '<', 'value': 0, 'format': red_fmt})

    # Current & Previous month values ku number format (admissions integer, mattha ellam 2 decimals)
    for r in range(len(kpi_df)):
        fmt = int_fmt if r == 0 else dec_fmt
        ws.write_number(4 + r, 1, kpi_df.loc[r, 'Current Month'], fmt)     # column B
        ws.write_number(4 + r, 2, kpi_df.loc[r, 'Previous Month'], fmt)    # column C

    # Key Insights section (KPI table kku keela)
    ws.write(11, 0, "Key Insights", section_fmt)
    for k, text in enumerate(insights):
        ws.write(12 + k, 0, f"{k + 1}. {text}")

    # ---------- Sheet 2: Medical Condition (with column chart) ----------
    ws = write_table(condition_df, 'Medical_Condition')
    bar = wb.add_chart({'type': 'column'})                                  # chart object
    n = len(condition_df)                                                   # rows count
    bar.add_series({'name': 'Billing',
                    'categories': ['Medical_Condition', 1, 0, n, 0],        # condition names
                    'values': ['Medical_Condition', 1, 2, n, 2]})           # billing values
    bar.set_title({'name': 'Billing by Medical Condition'})
    bar.set_y_axis({'name': 'Billing', 'num_format': '#,##0'})              # y axis title + format
    bar.set_legend({'none': True})                                          # single series, legend vendaam
    ws.insert_chart('G2', bar)

    # ---------- Sheet 3 to 7: Detail summaries ----------
    write_table(admission_df, 'Admission_Type')
    write_table(insurance_df, 'Insurance')
    write_table(age_df, 'Age_Group')
    write_table(medication_df, 'Medication')
    write_table(abnormal_df, 'Abnormal_by_Condition')

    # ---------- Sheet 8: Monthly Trend (with line chart) ----------
    ws = write_table(monthly_df, 'Monthly_Trend')
    line = wb.add_chart({'type': 'line'})                                   # line chart object
    m = len(monthly_df)                                                     # months count
    line.add_series({'name': 'Admissions',
                     'categories': ['Monthly_Trend', 1, 0, m, 0],           # month labels
                     'values': ['Monthly_Trend', 1, 1, m, 1]})              # admissions values
    line.set_title({'name': 'Monthly Admissions Trend (complete months)'})
    line.set_y_axis({'name': 'Admissions'})                                 # y axis title
    line.set_legend({'none': True})                                         # legend vendaam
    line.set_size({'width': 720, 'height': 320})
    ws.insert_chart('F2', line)

    # ---------- Sheet 9: Data Quality ----------
    write_table(dq_df, 'Data_Quality')

    # ---------- Sheet 10: Clean data (audit ku) ----------
    ws = write_table(cur, 'Clean_Data_CurrentMonth')
    ws.freeze_panes(1, 0)                                                   # header row freeze
    ws.autofilter(0, 0, len(cur), len(cur.columns) - 1)                     # filter buttons

    # ---------- Sheet 11: Validation (Excel formulas, report vs source data) ----------
    last_row = len(cur) + 1                                                 # last data row number in clean sheet
    bill_col = xl_col_to_name(cur.columns.get_loc('Billing_Amount'))        # Billing column letter
    los_col = xl_col_to_name(cur.columns.get_loc('Length_of_Stay'))         # LOS column letter
    ws = wb.add_worksheet('Validation')                                     # new sheet
    ws.write_row(0, 0, ['Check', 'Report Value', 'Source (Formula)', 'Difference', 'Status'], header_fmt)
    checks = [                                                              # (label, summary cell, source formula)
        ('Total Admissions', '=Summary!B5', f'=COUNTA(Clean_Data_CurrentMonth!A2:A{last_row})'),
        ('Total Billing', '=Summary!B6', f'=SUM(Clean_Data_CurrentMonth!{bill_col}2:{bill_col}{last_row})'),
        ('Avg Length of Stay', '=Summary!B8', f'=AVERAGE(Clean_Data_CurrentMonth!{los_col}2:{los_col}{last_row})'),
    ]
    for r, (label, rep, src) in enumerate(checks, start=1):
        ws.write(r, 0, label)                                               # check name
        ws.write_formula(r, 1, rep, dec_fmt)                                # report value
        ws.write_formula(r, 2, src, dec_fmt)                                # source value (live formula)
        ws.write_formula(r, 3, f'=ROUND(B{r + 1}-C{r + 1},2)', dec_fmt)     # difference
        ws.write_formula(r, 4, f'=IF(ABS(D{r + 1})<0.01,"OK","CHECK")')     # status
    ws.set_column(0, 4, 22)                                                 # column widths

# success message
print("Report created:", output_file)

Report created: Hospital_MIS_Report_2024-04.xlsx


In [14]:
# Excel file ah laptop ku download pannrom
files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>